# E-commerce Sales Analysis & Customer Segmentation (RFM)
**Project for Data Analyst Portfolio**
**Dataset:** Online Retail (10k+ records)


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import os

In [14]:
# 1. Load và Khám phá Dữ liệu Thô
# Load dữ liệu trực tiếp từ file data.csv
file_path = '../data/raw/data.csv'
df = pd.read_csv(file_path, encoding='ISO-8859-1')

print(f"Kích thước dữ liệu gốc: {df.shape}")
print("Các cột hiện có:", df.columns.tolist())
df.head()

Kích thước dữ liệu gốc: (541909, 8)
Các cột hiện có: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [15]:

# 2. Làm sạch Dữ liệu (Data Cleaning)
# 2.1 Xử lý Missing Values
# Xóa các dòng thiếu CustomerID vì không thể phân tích hành vi khách hàng
df.dropna(subset=['CustomerID'], inplace=True)

# 2.2 Xử lý Inconsistent Formats
# Loại bỏ các đơn hàng bị hủy (Quantity âm) và đơn giá bằng 0
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 2.3 Định dạng thời gian và Cập nhật )
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['InvoiceDate'] = df['InvoiceDate'] + pd.DateOffset(years=13, months=1)

# 2.4 Tính doanh thu cho từng dòng
df['Sales'] = df['Quantity'] * df['UnitPrice']

print(f"Kích thước sau khi làm sạch: {df.shape} (> 10,000 records - Pass)")


Kích thước sau khi làm sạch: (397884, 9) (> 10,000 records - Pass)


In [16]:

# 3. Phân tích RFM & Customer LTV
# Thiết lập ngày tham chiếu (Snapshot date)
snapshot_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Tổng hợp theo từng khách hàng
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'Sales': 'sum'                                           # Monetary (Historical LTV)
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm = rfm.reset_index()
rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346.0,327,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,76,4,1797.24
3,12349.0,20,1,1757.55
4,12350.0,313,1,334.40


In [17]:

# 4. Phân đoạn khách hàng (Segmentation)

# Chia khách hàng thành 5 nhóm dựa trên điểm số (Quintiles)
rfm["R_Score"] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm["F_Score"] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
rfm["M_Score"] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# Xác định Top 20% High-Value Clients (Dựa trên Monetary)
top_20_threshold = rfm['Monetary'].quantile(0.80)
rfm['Is_Top_20'] = np.where(rfm['Monetary'] >= top_20_threshold, 'High-Value', 'Standard')

# Tính toán Churn Rate (Nếu không mua hàng > 180 ngày)
rfm['Status'] = np.where(rfm['Recency'] > 180, 'Churned', 'Active')


In [18]:

# 5. Xuất dữ liệu cho Tableau
# Lưu file RFM tổng hợp
output_path = '../data/processed/'
if not os.path.exists(output_path):
    os.makedirs(output_path)

rfm.to_csv(f'{output_path}tableau_rfm_segmentation.csv', index=False)

# Lưu file giao dịch chi tiết (để vẽ biểu đồ thời gian)
df_final = df.merge(rfm[['CustomerID', 'Is_Top_20', 'Status']], on='CustomerID', how='left')
df_final.to_csv(f'{output_path}tableau_transactions_final.csv', index=False)


print("--- THÀNH CÔNG ---")
print(f"File 1: {output_path}tableau_rfm_segmentation.csv")
print(f"File 2: {output_path}tableau_transactions_final.csv")
print("Dữ liệu đã sẵn sàng để đưa vào Tableau và kết nối qua CustomerID.")

--- THÀNH CÔNG ---
File 1: ../data/processed/tableau_rfm_segmentation.csv
File 2: ../data/processed/tableau_transactions_final.csv
Dữ liệu đã sẵn sàng để đưa vào Tableau và kết nối qua CustomerID.
